# Experiment Tracking

The use of different tools to keep track of multiple experiments, what each of them tested and how they performed, allowing for best decision making.

Most common tools are [**TensorBoard**](https://www.tensorflow.org/tensorboard/get_started), [**Wandb**](https://wandb.ai/site/experiment-tracking) and [**MLFlow**](https://mlflow.org/).

In [14]:
# For this notebook to run with updated APIs, we need torch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install triton==3.9.0 --resume-retries 10
    !pip3 install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")

import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    %pip install -q torchinfo
    from torchinfo import summary

try:
    from torch.utils.tensorboard import SummaryWriter
except:
    print("[INFO] Couldn't find tensorboard... installing it.")
    %pip install -q tensorboard
    from torch.utils.tensorboard import SummaryWriter

torch version: 2.15.0+cu132
torchvision version: 0.29.2+cu132


#### Set up and get data

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RND_SEED = 42
torch.manual_seed(RND_SEED)
torch.cuda.manual_seed(RND_SEED)

from src.helper_functions import download_data

image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
print(image_path)

# Setup directories
train_dir = image_path / "train"
test_dir = image_path / "test"


"""Manual Transforms
# Setup ImageNet normalization levels (turns all images into similar distribution as ImageNet)
# https://www.image-net.org/
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# Create transform pipeline manually
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])
print(f"Manually created transforms: {manual_transforms}")
"""


# Setup pretrained weights (plenty of these available in torchvision.models)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# Get transforms from weights (these are the transforms that were used to obtain the weights)
automatic_transforms = weights.transforms()
print(f"Automatically created transforms: {automatic_transforms}")


# Create data loaders
from src.data_setup import create_img_dataloaders

train_dataloader, test_dataloader, class_names = create_img_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=automatic_transforms, # use manually created transforms
    batch_size=32
)

train_dataloader, test_dataloader, class_names


[INFO] data/pizza_steak_sushi directory exists, skipping download.
data/pizza_steak_sushi
Automatically created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)


(<torch.utils.data.dataloader.DataLoader at 0x7f6becba69f0>,
 ['pizza', 'steak', 'sushi'])

## Preparing a model

In [16]:
# Note: This is how a pretrained model would be created in torchvision > 0.13, it will be deprecated in future versions.
# model = torchvision.models.efficientnet_b0(pretrained=True).to(device) # OLD

# Download the pretrained weights for EfficientNet_B0
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # NEW in torchvision 0.13, "DEFAULT" means "best weights available"

# Setup the model with the pretrained weights and send it to the target device
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# View the output of the model

for p in model.features.parameters():
    p.requires_grad = False

prev_classifier = model.classifier

from torch.nn import Sequential, Dropout, Linear

model.classifier = Sequential(
    Dropout(p=0.2, inplace=True),
    Linear(in_features=1280, out_features=len(class_names))
)

prev_classifier, model.classifier

(Sequential(
   (0): Dropout(p=0.2, inplace=True)
   (1): Linear(in_features=1280, out_features=1000, bias=True)
 ),
 Sequential(
   (0): Dropout(p=0.2, inplace=True)
   (1): Linear(in_features=1280, out_features=3, bias=True)
 ))

In [17]:
from torchinfo import summary
summary(
    model=model,
    input_size=(1, 3, 224, 224), # batch, colors, height and width
    verbose=0,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 3]               --                   Partial
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1280, 7, 7]      --                   False
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   False
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    (864)                False
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    (64)                 False
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1, 1

In [18]:
from src.training import train_model_with_batches
from src.helper_functions import accuracy_fn, set_seeds

set_seeds(RND_SEED)

results = train_model_with_batches(
    model=model,
    train_data=train_dataloader,
    test_data=test_dataloader,
    loss_fn=nn.CrossEntropyLoss(),
    optimizer_fn=torch.optim.Adam,
    lr=0.001,
    N_EPOCHS=5,
    RND_SEED=RND_SEED,
    eval_fn=accuracy_fn,
    is_classification=True,
    verbose=1
)

Epoch: 1
 - Train Loss: 1.0750
 - Test Loss:  0.8618
 - Test Eval: 76.33%


/home/maykon/.pyenv/versions/.colab_env/lib/python3.12/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/maykon/.pyenv/versions/.colab_env/lib/python3.12/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Epoch: 2
 - Train Loss: 0.8802
 - Test Loss:  0.7511
 - Test Eval: 84.47%
Epoch: 3
 - Train Loss: 0.7837
 - Test Loss:  0.6427
 - Test Eval: 85.51%
Epoch: 4
 - Train Loss: 0.6414
 - Test Loss:  0.6600
 - Test Eval: 82.48%
Epoch: 5
 - Train Loss: 0.6643
 - Test Loss:  0.6635
 - Test Eval: 77.37%
Total time on cuda:0: 29.886 seconds


  0%|          | 0/3 [00:00<?, ?it/s]

# Using TensorBoard

It's the TensorFlow visualization toolkit fro tracking ML experiments data: https://www.tensorflow.org/tensorboard

Interacts naturally with pytorch through [SummaryWriter](https://pytorch.org/docs/stable/tensorboard.html)

The SummaryWriter() class stores the model's results in a directory called runs/ in TensorBoard format by default (can be changed on the initialization).



In [19]:
# Load the TensorBoard notebook extension
%reload_ext tensorboard

# change the automatic log directory
%tensorboard --logdir runs

# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter()

Reusing TensorBoard on port 6007 (pid 41106), started 0:00:53 ago. (Use '!kill 41106' to kill it.)